# Create a panel for all historical companies in Denmark using Danish CVR

This notebook demonstrates how to get all companies in Denmark, active and disolved, and create a panel dataset. 

## The overall steps coded below are:

### 1. Download the all the data using the python scripts in this repo and store them as `parquet` files.

**IMPORTANT**: If you have not downloaded the data you will need to run the `data_extraction` scripts. This step take some time to request the data (API request limits) and requires a fairly good machine (need to be able to merge into 2M rows parquet files).

The strip download all Danish companies with their full history at a given founding year. For example: `python data_extraction/src/virksomhed_api_call.py --founding-years 1991 2000` downloads all the companies founded from 1991 to 2000. All years included.

The oldest CVR in Denmark is University of Copenhagen at  1300, but there are no records until 1735 in the API - so we start from companies founded from 1700 onwards. 

Fell free to check the source code at `data_extraction/src/virksomhed_api_call.py`. 


### 2. Get from the `main` dataset: the last company name and founding date

  - `Vrvirksomhed_cvrNummer` (cvr_number)
  - `Vrvirksomhed_virksomhedMetadata_nyesteNavn_navn` (latest_name)
  - `Vrvirksomhed_virksomhedMetadata_stiftelsesDato` (founding_date)

### 3. Get from `livsforloeb` (company lifecycle) dataset: the activity time stamps

  - `cvrNummer` (cvr_number)
  - `gyldigFra` (valid_from)
  -	`gyldigTil` (valid_to)

This will give from which day to which day the company was active. 

It can include `gyldigFra` *after* `31-12-2025` since a company can be founded in 2025 December and signed to be active in 2026 January, for example.

Values `gyldigTil == None` means that the company is active at the time of the data extraction ("No  end validity"). Therefore, every time you run this query you likely have different set of companies, as companies close daily.

Since I wanted to have a fix point (1700-2025):

  - I deleted any company with `gyldigFra` *after* `31-12-2025`. This means that companies created in 2025 but signed to be active in 2026 are not included.
  - I considered "Active" any company closed in 2026. Closing in 2026 means that they were active in 2025. Effectively, this can be done setting `gyldigFra == None` (active) for companies with a closing date *after* `31-12-2025`.

### 4. Get from `virksomhedsform` (business form) dataset: the company type: 

This is relevant for research since there are some reserach questions that need to limit the data to APS companies only, for example. 

  - `kortBeskrivelse` (short_company_legal_name)
  - `langBeskrivelse` (long_company_legal_name)
  - `gyldigFra` (valid_from)
  -	`gyldigTil` (valid_to)

### 5. Merge datasets (no panel)

Merging all the data together.

Notice the `cvrNummer` is non unique for `livsforloeb` and `virksomhedsform` datasets, and therefore for the final dataset. The same company can appear multiple times as:

- Companies re-open at different times using the same CVR number.
- Companies can  move from one city to another (e.g. https://cvrapi.dk/api?search=12397399&country=dk)
- Companies change their legal structure, or shut down for some years (e.g. https://cvrapi.dk/api?search=10000173&country=dk). These companies have different `productionunits` ID (identification similar as CVR) for every time they re-open.


In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

pd.set_option('display.max_columns', None)
load_dotenv()

PROJECT_HOME_PATH = os.getenv("PROJECT_HOME_PATH")
COMPANY_DATA_FOLDER_PATH = os.getenv("COMPANY_DATA_FOLDER_PATH")

os.chdir(PROJECT_HOME_PATH)

# Import translation utilities
from utils.translations import COLUMN_TRANSLATIONS, VALUE_TRANSLATIONS

## 1. Download the all the data using the python scripts in this repo and store them as `parquet` files.

In [2]:
# Expect hours downloading data
#!python data_extraction/src/virksomhed_api_call.py --founding-years 1700 1990
#!python data_extraction/src/virksomhed_api_call.py --founding-years 1991 2000
#!python data_extraction/src/virksomhed_api_call.py --founding-years 2001 2010
#!python data_extraction/src/virksomhed_api_call.py --founding-years 2011 2020
#!python data_extraction/src/virksomhed_api_call.py --founding-years 2021 2025


## 2. Get from the `main` dataset the last name and founding date

In [150]:
def merge_parquets(dataset, fields=None,company_data_folder_path=COMPANY_DATA_FOLDER_PATH):

  df = pd.DataFrame()
  years = ["1700_1990", "1991_2000", "2001_2010", "2011_2020", "2021_2025"]
  for year in years:
    file_path = f"{company_data_folder_path}/virksomhed_founded_{year}_{dataset}.parquet"
    df_year = pd.read_parquet(file_path, columns=fields)
    df = pd.concat([df, df_year])

  df.reset_index(drop=True, inplace=True)

  return df

In [151]:
main = merge_parquets(dataset="main", fields=None)

In [127]:
# fields = ["Vrvirksomhed_cvrNummer",
#           "Vrvirksomhed_virksomhedMetadata_nyesteNavn_navn",
#           "Vrvirksomhed_virksomhedMetadata_stiftelsesDato"]
# main = merge_parquets(dataset="main", fields=None)

#main.rename(columns=COLUMN_TRANSLATIONS)


,search_score,search_id,search_type,search_index,cvr_number,industry_responsibility_code,advertising_protected,latest_name,latest_name_valid_from,latest_name_valid_to,latest_name_last_updated,latest_legal_form_code,latest_legal_form_short,latest_legal_form_long,latest_legal_form_data_provider,latest_legal_form_valid_from,latest_legal_form_valid_to,latest_legal_form_last_updated,latest_address_country_code,latest_address_free_text,latest_address_road_code,latest_address_municipality_code,latest_address_municipality_name,latest_address_municipality_valid_from,latest_address_municipality_valid_to,latest_address_municipality_last_updated,latest_address_house_number_from,latest_address_id,latest_address_last_validated,latest_address_house_number_to,latest_address_letter_from,latest_address_letter_to,latest_address_floor,latest_address_side_door,latest_address_care_of_name,latest_address_po_box,latest_address_street_name,latest_address_city_name,latest_address_postal_code,latest_address_postal_district,latest_address_valid_from,latest_address_valid_to,latest_address_last_updated,latest_main_industry_code,latest_main_industry_text,latest_main_industry_valid_from,latest_main_industry_valid_to,latest_main_industry_last_updated,latest_secondary_industry1_json,latest_secondary_industry2_json,latest_secondary_industry3_json,latest_status_json,number_of_p_units,latest_annual_employment_year,latest_annual_count_including_owners,latest_annual_full_time_equivalents,latest_annual_employee_count,latest_annual_employment_last_updated,latest_annual_interval_code_including_owners,latest_annual_interval_code_fte,latest_annual_interval_code_employees,latest_quarterly_employment_json,latest_monthly_employment_json,latest_first_monthly_employment_json,company_status,founding_date,effective_date,internal_id,error_registered,data_access,unit_number,unit_type,last_loaded,last_updated,error_on_loading,nearest_future_date,error_description,change_actor,latest_main_industry_json,latest_annual_employment_json,latest_quarterly_employment_year,latest_quarterly_employment_quarter,latest_quarterly_full_time_equivalents,latest_quarterly_employee_count,latest_quarterly_employment_last_updated,latest_quarterly_interval_code_fte,latest_quarterly_interval_code_employees,latest_secondary_industry1_code,latest_secondary_industry1_text,latest_secondary_industry1_valid_from,latest_secondary_industry1_valid_to,latest_secondary_industry1_last_updated,latest_monthly_employment_year,latest_monthly_employment_month,latest_monthly_full_time_equivalents,latest_monthly_employee_count,latest_monthly_employment_last_updated,latest_monthly_interval_code_fte,latest_monthly_interval_code_employees,latest_first_monthly_employment_year,latest_first_monthly_employment_month,latest_first_monthly_full_time_equivalents,latest_first_monthly_employee_count,latest_first_monthly_employment_last_updated,latest_first_monthly_interval_code_fte,latest_first_monthly_interval_code_employees,confidential_enriched,latest_status_code,latest_status_text,latest_status_credit_info_code,latest_status_credit_info_text,latest_status_valid_from,latest_status_valid_to,latest_status_last_updated,latest_secondary_industry2_code,latest_secondary_industry2_text,latest_secondary_industry2_valid_from,latest_secondary_industry2_valid_to,latest_secondary_industry2_last_updated,latest_secondary_industry3_code,latest_secondary_industry3_text,latest_secondary_industry3_valid_from,latest_secondary_industry3_valid_to,latest_secondary_industry3_last_updated,latest_legal_form_json,latest_address_municipality_json,latest_address_json,latest_name_json
0,None,4001150153,_doc,cvr-v-20220630,50506150,0.0,False,INTERESSENTSKABET AF 1. APRIL 1983 I/S,1983-04-01,2003-06-30,2013-11-22T22:12:17.000+01:00,30.0,I/S,Interessentskab,T&S,1983-04-01,2003-06-30,2013-11-22T22:00:13.000+01:00,DK,None,9584.0,219.0,HILLERØD,None,2006-12-31,1999-10-15T00:00:00.000+02:00,11.0,0a3f50a9-e233-32b8-e044-0003ba298018,2021-02-20T01:04:55.506+01:00,NaN,N

## 3. Get from `livsforloeb` (lifecycle) dataset the activity time stamps

In [110]:
fields = ["cvrNummer", "gyldigFra",	"gyldigTil"]
lifecycle = merge_parquets(dataset="livsforloeb", fields=fields)

In [111]:
lifecycle

,cvrNummer,gyldigFra,gyldigTil
0,50506150,1983-04-01,2003-06-30
1,57981512,1976-06-22,1990-01-02
2,50142019,1974-01-01,2000-12-31
3,58282812,1976-01-14,2006-08-22
4,11517536,1967-07-03,2019-06-26
...,...,...,...
2468306,45478041,2025-03-19,None
2468307,45378799,2025-02-04,None
2468308,45586626,2025-04-28,None
2468309,45585840,2025-05-01,None


In [112]:
# Sample of companies that closed operations in 1st Jan 2026
lifecycle[lifecycle["gyldigTil"] == "2026-01-01"].sample(5)

,cvrNummer,gyldigFra,gyldigTil
563280,17743406,2026-01-01,2026-01-01
2108185,33674236,2026-01-01,2026-01-01
1735343,40992936,2022-12-30,2026-01-01
1739702,39052172,2017-10-26,2026-01-01
1736878,37888877,2022-12-29,2026-01-01


In [113]:
# Sample of companies that are active as of current date (13 May 2026)
# You can check the CVRs manually using this website: https://cvrapi.dk/
lifecycle[lifecycle["gyldigTil"].isna()].sample(5)

,cvrNummer,gyldigFra,gyldigTil
2004511,36995270,2015-08-14,None
1871282,40201424,2019-01-25,None
1210402,31315131,2017-07-04,None
2094463,39895323,2018-09-27,None
1826361,39201305,2024-01-17,None


In [114]:
def set_temporal_cutoff(data,cutoff_date):

  df = data.copy()
  df["gyldigTil"] = pd.to_datetime(df["gyldigTil"], errors='coerce')
  #  `gyldigFra == None` (active) for companies with a closing date *after* `31-12-2025`.
  df["gyldigTil"] = df["gyldigTil"].where(df["gyldigTil"] <= pd.Timestamp(cutoff_date), None)

  return df

In [115]:
# Setting the cutoff
lifecycle = set_temporal_cutoff(lifecycle, cutoff_date="2025-12-31")

In [116]:
# After setting closed companies in 2026 as active (None) for 2025.
test_cvrs = [12354134, 12397399, 44573350, 17743406, 25625714,
             11088988, 86691450, 28815212, 12992742, 95589855]
lifecycle[lifecycle["cvrNummer"].isin(test_cvrs)].sort_values(["cvrNummer", "gyldigFra"])
# As an example, 12992742 closed multiple times, but since the last one is in 2026 counts as active (None)

,cvrNummer,gyldigFra,gyldigTil
210922,11088988,1986-03-01,NaT
244010,12354134,1988-01-01,NaT
244377,12397399,1983-04-19,2022-03-12
244378,12397399,2022-04-01,NaT
235565,12992742,1972-09-15,2008-06-30
235566,12992742,2023-06-01,2023-08-30
235567,12992742,2025-11-01,NaT
563279,17743406,1994-05-03,2004-12-31
563280,17743406,2026-01-01,NaT
568973,25625714,2000-09-18,NaT


In [118]:
lifecycle[lifecycle["gyldigFra"] > "2025-12-31"]

,cvrNummer,gyldigFra,gyldigTil
255991,12771347,2026-01-17,NaT
258858,86133652,2026-01-01,NaT
258860,60147159,2026-01-01,NaT
262473,17095552,2026-01-01,NaT
262483,25090055,2026-01-01,NaT
...,...,...,...
2466373,43746901,2026-01-20,NaT
2466377,43631640,2026-01-20,NaT
2466957,43485350,2026-01-21,NaT
2467366,44661756,2026-01-19,NaT


In [119]:
# Droping any observation for companies active in 2026
lifecycle = lifecycle[lifecycle["gyldigFra"] <= "2025-12-31"]
lifecycle[lifecycle["gyldigFra"] > "2025-12-31"]

,cvrNummer,gyldigFra,gyldigTil


## 4. Get from `virksomhedsform` the company legal status fields

In [120]:
fields = ["cvrNummer", "kortBeskrivelse", "langBeskrivelse", "gyldigFra", "gyldigTil"]
legal_form = merge_parquets(dataset="virksomhedsform", fields=fields)

In [122]:
# The future validity of the company type is not important for gathering companies
# It does not affect the cutoff
pd.to_datetime(legal_form["gyldigTil"], errors='coerce').max()

Timestamp('2027-01-06 00:00:00')

## 5. Merge datasets (no panel)

In [123]:
def translate_dfs(main=main,
                  lifecycle=lifecycle,
                  legal_form=legal_form,
                  timestamp="31-12-2025"):

  # Main dataset
  main = (main.rename(columns={"gyldigFra":"active_company_from", "gyldigTil": "active_company_to"})
              .rename(columns=COLUMN_TRANSLATIONS) )
  main["timestamp"] = timestamp

  # Lifecycle dataset
  lifecycle = lifecycle.rename(columns=COLUMN_TRANSLATIONS)

  # Legal status
  legal_form = (legal_form.rename(columns={"gyldigFra":"legal_type_valid_from", "gyldigTil": "legal_type_valid_until"})
                          .rename(columns=COLUMN_TRANSLATIONS) )
  

  df_merge = (main.merge(lifecycle, on="cvr_number", how="outer")
                  .merge(legal_form, on="cvr_number", how="outer"))
    
  return df_merge
    
translate_dfs()

,cvr_number,latest_name,founding_date,timestamp,valid_from,valid_to,short_company_legal_name,long_company_legal_name,legal_type_valid_from,legal_type_valid_until
0,10000009,YELLOW ApS,1999-10-12,31-12-2025,1999-10-12,2001-12-11,APS,Anpartsselskab,1999-10-12,2001-12-11
1,10000025,WATERFRONT CONNECTION ApS,1999-10-13,31-12-2025,1999-10-13,NaT,APS,Anpartsselskab,1999-10-13,None
2,10000068,"STUDENTCONSULTING, FILIAL AF SVERIGES STU...",1999-10-18,31-12-2025,1999-10-18,2001-12-20,FAS,"Filial af udenlandsk aktieselskab, kommanditak...",1999-10-18,2001-12-20
3,10000106,TRANBJERG TAGDÆKNING V/JOHN HARTM...,1985-07-11,31-12-2025,1985-07-11,2007-06-30,ENK,Enkeltmandsvirksomhed,1985-07-11,2007-06-30
4,10000122,DIGITAL CENTER FYN ApS,1999-10-14,31-12-2025,1999-10-14,2002-08-15,APS,Anpartsselskab,1999-10-14,2002-08-15
...,...,...,...,...,...,...,...,...,...,...
3123683,99995653,Skovly FerieCenter,1986-07-01,31-12-2025,1986-07-01,2004-01-01,ENK,Enkeltmandsvirksomhed,2019-05-28,None
3123684,99995653,Skovly FerieCenter,1986-07-01,31-12-2025,2019-05-28,NaT,ENK,Enkeltmandsvirksomhed,1986-07-01,2004-01-01
3123685,99995653,Skovly FerieCenter,1986-07-01,31-12-2025,2019-05-28,NaT,ENK,Enkeltmandsvirksomhed,2019-05-28,None
3123686,99997451,SCHÆFERKLUBBEN KREDS 52,1986-06-10,31-12-2025,1986-06-10,NaT,FOR,Forening,1986-06-10,None


In [17]:
def translate_dfs(main=main,lifecycle=lifecycle,legal_form=legal_form):

  df_1 = main.rename(columns=COLUMN_TRANSLATIONS)
  df_2 = lifecycle.rename(columns={"cvrNummer":"cvr_number",
                         "gyldigFra":"company_from",
                         "gyldigTil": "company_to"})
  df_3 = (legal_form
           .replace(VALUE_TRANSLATIONS)
           .rename(columns={"cvrNummer":"cvr_number",
                            "kortBeskrivelse": "short_company_legal_name",
                            "langBeskrivelse": "long_company_legal_name",
                            "gyldigFra":"legal_form_from",
                            "gyldigTil": "legal_form_to"}))

  merged = df_2.merge(df_3, on="cvr_number", how="outer")

  print(f"Lifecycle rows: {len(df_2):,}")
  print(f"Legal form rows: {len(df_3):,}")
  print(f"Merged rows: {len(merged):,}")

  return merged


df = translate_dfs()

Lifecycle rows: 2,466,953
Legal form rows: 2,543,121
Merged rows: 3,123,688


In [18]:
df

,cvr_number,company_from,company_to,short_company_legal_name,long_company_legal_name,legal_form_from,legal_form_to
0,10000009,1999-10-12,2001-12-11 00:00:00,APS,private_limited_company,1999-10-12,2001-12-11
1,10000025,1999-10-13,None,APS,private_limited_company,1999-10-13,None
2,10000068,1999-10-18,2001-12-20 00:00:00,FAS,branch_of_foreign_public_limited_company,1999-10-18,2001-12-20
3,10000106,1985-07-11,2007-06-30 00:00:00,ENK,sole_proprietorship,1985-07-11,2007-06-30
4,10000122,1999-10-14,2002-08-15 00:00:00,APS,private_limited_company,1999-10-14,2002-08-15
...,...,...,...,...,...,...,...
3123683,99995653,1986-07-01,2004-01-01 00:00:00,ENK,sole_proprietorship,2019-05-28,None
3123684,99995653,2019-05-28,None,ENK,sole_proprietorship,1986-07-01,2004-01-01
3123685,99995653,2019-05-28,None,ENK,sole_proprietorship,2019-05-28,None
3123686,99997451,1986-06-10,None,FOR,association,1986-06-10,None


In [19]:
# Check the test companies from before
test_cvrs = [12354134, 12397399, 44573350, 17743406, 25625714,
             11088988, 86691450, 28815212, 12992742, 95589855]
df[df["cvr_number"].isin(test_cvrs)]

,cvr_number,company_from,company_to,short_company_legal_name,long_company_legal_name,legal_form_from,legal_form_to
41107,11088988,1986-03-01,None,I/S,general_partnership,1986-03-01,2026-03-15
76452,12354134,1988-01-01,None,ENK,sole_proprietorship,1988-01-01,2026-01-01
77655,12397399,1983-04-19,2022-03-12 00:00:00,ENK,sole_proprietorship,1983-04-19,2018-12-31
77656,12397399,1983-04-19,2022-03-12 00:00:00,PMV,personally_owned_small_business,2019-01-01,2022-03-12
77657,12397399,1983-04-19,2022-03-12 00:00:00,PMV,personally_owned_small_business,2022-04-01,2026-01-01
77658,12397399,2022-04-01,None,ENK,sole_proprietorship,1983-04-19,2018-12-31
77659,12397399,2022-04-01,None,PMV,personally_owned_small_business,2019-01-01,2022-03-12
77660,12397399,2022-04-01,None,PMV,personally_owned_small_business,2022-04-01,2026-01-01
96140,12992742,1972-09-15,2008-06-30 00:00:00,ENK,sole_proprietorship,1972-09-15,2008-06-30
96141,12992742,1972-09-15,2008-06-30 00:00:00,ENK,sole_proprietorship,2023-06-01,2023-08-30
